# Unit 5 practical: a RAG chatbot over your own documents (real models)
> **Status: written but NOT executed** while the course was authored (no GPU, no embedding model, no language model on the authoring machine).
> Both models named below were confirmed to exist on the Hugging Face hub. Run on Google Colab; if a cell fails, a library version has likely changed.

You will: (1) upload your own PDF(s), (2) chunk them with the course code, (3) retrieve with **BM25**, a **neural embedding model** and a **hybrid**, (4) measure each on questions *you* write, (5) answer with a small chat model using only the retrieved text.

In [ ]:
!pip -q install sentence-transformers transformers accelerate faiss-cpu pypdf
!wget -q https://krunalpanchal1995.github.io/banu.github.io/downloads/u5/u5-code.zip -O u5.zip && unzip -q -o u5.zip
import sys; sys.path.insert(0, "u5-code/codes/rag/rag_np"); import rag_np as R
import numpy as np

## 1. Upload and chunk your documents

In [ ]:
from google.colab import files
from pypdf import PdfReader
import re, io
up = files.upload()                      # choose one or more PDFs or .txt files
docs = {}
for name, data in up.items():
    text = "".join(p.extract_text() or "" for p in PdfReader(io.BytesIO(data)).pages) if name.lower().endswith(".pdf") else data.decode("utf-8", "ignore")
    docs[name] = re.sub(r"\s+", " ", text).strip()
chunks = R.chunk_corpus(docs, size=600, overlap=120)
T = [c["text"] for c in chunks]
print(len(chunks), "chunks from", len(docs), "document(s)")

## 2. Three retrievers

In [ ]:
bm25 = R.BM25(T)
from sentence_transformers import SentenceTransformer
emb = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
E = emb.encode(T, normalize_embeddings=True, show_progress_bar=False)      # (n_chunks, 384)
import faiss
index = faiss.IndexFlatIP(E.shape[1]); index.add(E.astype("float32"))      # exact inner-product search; try faiss.IndexIVFFlat for an IVF index
def dense_scores(q):
    v = emb.encode([q], normalize_embeddings=True)[0]
    return E @ v
def hybrid(q):
    return R.as_ranking(R.rrf([R.rank(bm25.scores(q)), R.rank(dense_scores(q))]))

## 3. Measure them on YOUR questions
Write 15+ questions. For each, `gold` must be a short phrase that occurs in exactly one place in your documents. 15 questions is **too few to rank retrievers** (Section 3.3.3 of the PDF): look at the paired interval.

In [ ]:
queries = [
    {"q": "REPLACE with a question about your document", "gold": "REPLACE with a phrase that answers it"},
    # ... add more
]
results = {}
for name, f in [("BM25", bm25.scores), ("dense (MiniLM)", dense_scores), ("hybrid (RRF)", hybrid)]:
    r = R.evaluate(f, chunks, queries, ks=(1, 3, 5))
    results[name] = r
    print(f"{name:16s} recall@1 {r['recall@1']:.2f}  @3 {r['recall@3']:.2f}  @5 {r['recall@5']:.2f}  MRR {r['mrr']:.2f}")

## 4. Answer from the evidence with a small chat model

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
name = "Qwen/Qwen2.5-0.5B-Instruct"
tok = AutoTokenizer.from_pretrained(name); model = AutoModelForCausalLM.from_pretrained(name, torch_dtype="auto", device_map="auto")
def answer(question, k=3, min_cov=0.4):
    order = hybrid(question)
    if R.query_coverage(question, chunks[int(order[0])]["text"]) < min_cov:
        return "I don't know: the documents do not seem to contain this.", []
    ctx = R.build_context(chunks, order[:k], budget_chars=2500)
    sources = [f"[{j+1}] {chunks[i]['text']}" for j, i in enumerate(ctx)]
    msgs = [{"role": "system", "content": "Answer ONLY from the numbered passages. Cite the passage number. If the passages do not contain the answer, say you do not know."},
            {"role": "user", "content": "Passages:\n" + "\n".join(sources) + "\n\nQuestion: " + question}]
    ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to(model.device)
    out = model.generate(ids, max_new_tokens=200, do_sample=False)
    return tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True).strip(), ctx
print(answer(queries[0]["q"]))

## Your turn
1. Ask a question your documents do **not** answer. Does the abstention rule work? Change `min_cov` and measure the trade-off (Section 3.4.3).
2. Change `size` and `overlap` in step 1 and compare on an **equal-cost** axis (characters retrieved), as in Figure 2 of the PDF.
3. Replace `faiss.IndexFlatIP` by `faiss.IndexIVFFlat` and measure recall against the exact index for several `nprobe` values (Section 3.3.4).
4. Is the generated answer *grounded*? For five answers, check by hand that each claim appears in the cited passage.